<a href="https://colab.research.google.com/github/Rohita-G/NLP_CLASS/blob/main/NLP_HW_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q1 — Regex

In [2]:
import re

# Q1.1
text = "Hello World python Data"
pattern = r'\b[A-Z][a-z]*\b'

print("Q1.1:", re.findall(pattern, text))


# Q1.2
text = "hello world don't well-known Python"
pattern = r"\b[a-z][\w'-]*\b"

print("Q1.2:", re.findall(pattern, text))


# Q1.3
text = "12 123 1,234 -45 3.14 1.5e10"
pattern = r'[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?'

print("Q1.3:", re.findall(pattern, text))


# Q1.4
text = "Contact me at test@gmail.com or abc123@yahoo.com"
pattern = r'\b[\w.-]+@[\w.-]+\.[A-Za-z]{2,}\b'

print("Q1.4:", re.findall(pattern, text))


# Q1.5
text = "Visit https://www.google.com or http://example.org"
pattern = r'https?://[^\s]+'

print("Q1.5:", re.findall(pattern, text))


# Q1.6
text = """How are you?
Are you coming?
This is a statement.
Really?"""

pattern = r'\?\s*[\)"’\]]*\s*$'

print("Q1.6:", re.findall(pattern, text, re.MULTILINE))

Q1.1: ['Hello', 'World', 'Data']
Q1.2: ['hello', 'world', "don't", 'well-known']
Q1.3: ['12', '123', '1,234', '-45', '3.14', '1.5e10']
Q1.4: ['test@gmail.com', 'abc123@yahoo.com']
Q1.5: ['https://www.google.com', 'http://example.org']
Q1.6: ['?', '?', '?']


#Q2.1 — Manual BPE on Toy Corpus

In [3]:
from collections import Counter

corpus = [
    "low", "low", "low", "low", "low",
    "lowest", "lowest",
    "newer", "newer", "newer", "newer", "newer", "newer",
    "wider", "wider", "wider",
    "new", "new"
]

# Add end-of-word marker
words = [list(word) + ["_"] for word in corpus]

# Count bigrams
def get_pair_counts(words):
    pairs = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += 1

    return pairs


pair_counts = get_pair_counts(words)

print("Bigram counts:")
for pair, count in pair_counts.items():
    print(pair, ":", count)

Bigram counts:
('l', 'o') : 7
('o', 'w') : 7
('w', '_') : 7
('w', 'e') : 8
('e', 's') : 2
('s', 't') : 2
('t', '_') : 2
('n', 'e') : 8
('e', 'w') : 8
('e', 'r') : 9
('r', '_') : 9
('w', 'i') : 3
('i', 'd') : 3
('d', 'e') : 3


#Q2.2 — Code a Mini-BPE Learner

In [4]:
from collections import Counter

# Toy corpus
corpus = [
    "low", "low", "low", "low", "low",
    "lowest", "lowest",
    "newer", "newer", "newer", "newer", "newer", "newer",
    "wider", "wider", "wider",
    "new", "new"
]

# Add end-of-word marker
words = [list(word) + ["_"] for word in corpus]


# Count adjacent token pairs
def get_pair_counts(words):
    pairs = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += 1

    return pairs


# Merge selected pair
def merge_pair(words, pair):
    new_words = []

    for word in words:
        merged_word = []
        i = 0

        while i < len(word):

            if (
                i < len(word) - 1
                and word[i] == pair[0]
                and word[i + 1] == pair[1]
            ):
                merged_word.append(word[i] + word[i + 1])
                i += 2

            else:
                merged_word.append(word[i])
                i += 1

        new_words.append(merged_word)

    return new_words


# Initial vocabulary
vocab = set(token for word in words for token in word)

print("Initial vocabulary size:", len(vocab))


# Learn 3 BPE merges
for step in range(1, 4):

    pair_counts = get_pair_counts(words)

    top_pair, count = pair_counts.most_common(1)[0]

    new_token = top_pair[0] + top_pair[1]

    words = merge_pair(words, top_pair)

    vocab.add(new_token)

    print("\nStep", step)
    print("Top pair:", top_pair)
    print("Count:", count)
    print("New token:", new_token)
    print("Vocabulary size:", len(vocab))

Initial vocabulary size: 11

Step 1
Top pair: ('e', 'r')
Count: 9
New token: er
Vocabulary size: 12

Step 2
Top pair: ('er', '_')
Count: 9
New token: er_
Vocabulary size: 13

Step 3
Top pair: ('n', 'e')
Count: 8
New token: ne
Vocabulary size: 14


#Q2.2 — Word Segmentation
Using the BPE tokens learned in the previous steps:

new → ne + w + _

newer → ne + w + er_

lowest → l + o + w + e + s + t + _

widest → w + i + d + e + s + t + _

newestest → ne + w + e + s + t + e + s + t + _

#Q2.2 — OOV Explanation
BPE helps solve the OOV problem by breaking unknown words into smaller known parts. Even if the complete word was not seen before, its smaller subwords can still be used. For example, newer can be split into ne + w + er_. The token er_ can be useful because -er can occur as a suffix in English. This allows the model to handle new words without needing a separate token for every word. Overall, subword tokenization gives the model more flexibility with new or rare words.

#Q2.3 — Your Language / English(Q2.3 — Train BPE)
Paragraph:

I study data science at university. I learn Python and practice coding every day. My classes teach useful skills for data analysis and machine learning. I enjoy building small projects because projects help me understand new ideas. Learning regularly makes difficult topics easier and gives me confidence.


In [5]:
from collections import Counter

text = """I study data science at university.
I learn Python and practice coding every day.
My classes teach useful skills for data analysis and machine learning.
I enjoy building small projects because projects help me understand new ideas.
Learning regularly makes difficult topics easier and gives me confidence."""

# Convert text to lowercase
text = text.lower()

# Split into words
corpus = text.split()

# Add end-of-word marker
words = [list(word) + ["_"] for word in corpus]


def get_pair_counts(words):
    pairs = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += 1

    return pairs


def merge_pair(words, pair):
    new_words = []

    for word in words:
        new_word = []
        i = 0

        while i < len(word):

            if (
                i < len(word) - 1
                and word[i] == pair[0]
                and word[i + 1] == pair[1]
            ):
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        new_words.append(new_word)

    return new_words


# Initial vocabulary
vocab = set(token for word in words for token in word)

merges = []

# Learn 30 merges
for step in range(30):

    pair_counts = get_pair_counts(words)

    if not pair_counts:
        break

    top_pair, count = pair_counts.most_common(1)[0]

    new_token = top_pair[0] + top_pair[1]

    words = merge_pair(words, top_pair)

    vocab.add(new_token)

    merges.append((top_pair, count, new_token))


print("Initial vocabulary size:", len(vocab))

print("\nFive most frequent merges:")
for pair, count, token in merges[:5]:
    print(pair, "→", token, "| Count:", count)


# Find longest tokens
tokens_sorted = sorted(vocab, key=lambda x: len(x), reverse=True)

print("\nFive longest tokens:")
for token in tokens_sorted[:5]:
    print(token, "| Length:", len(token))

Initial vocabulary size: 55

Five most frequent merges:
('s', '_') → s_ | Count: 8
('e', '_') → e_ | Count: 6
('e', 'a') → ea | Count: 6
('y', '_') → y_ | Count: 5
('.', '_') → ._ | Count: 5

Five longest tokens:
data_ | Length: 5
learn | Length: 5
and_ | Length: 4
data | Length: 4
lear | Length: 4


#Q2.3 — Word Segmentation

Data → data_

Learning → learn + ing_

Practice → pr + a + ct + i + c + e_

University → u + n + i + v + er + si + t + y_

Regularly → r + e + g + ul + a + r + l + y_

# Q2.3 — Reflection
The BPE model learned different parts of words such as learn, ing_, and_, and data_. Some tokens are complete words, while others are parts of words or suffixes. One advantage of BPE is that it can handle rare and new words by breaking them into smaller known parts. Another advantage is that it reduces the number of unknown words. One disadvantage is that some subwords do not have a clear meaning by themselves. Another disadvantage is that some words can be split into many small pieces. Overall, BPE is useful because it can represent both common words and smaller parts of words.

# Q3 — Bayes Rule
P(c) → The probability of a class before looking at the document. It is called the prior probability.

P(d | c) → The probability of seeing document d if it belongs to class c. It tells us how likely the document is for that class.

P(c | d) → The probability that the document belongs to class c after seeing the document. It is called the posterior probability.

The denominator P(d) can be ignored when comparing classes because the document is the same for every class. Therefore, P(d) has the same value for every class. We only need to compare P(d | c)P(c) to determine which class has the highest probability.

# Q4 — Add-1 Smoothing
Given:

P(-) = 3/5
P(+) = 2/5
Vocabulary size V = 20
Total negative tokens = 14

1. Denominator:

Denominator = total tokens + V
            = 14 + 20
            = 34


2. P(predictable | -):

P(predictable | -)
= (2 + 1) / (14 + 20)
= 3/34
≈ 0.0882


3. P(fun | -):

P(fun | -)
= (0 + 1) / (14 + 20)
= 1/34
≈ 0.0294

# Q5 — Tokenization
### Q5.1 — Telugu Paragraph


నేను ప్రతిరోజూ కాలేజీకి వెళ్తాను. అక్కడ నేను కొత్త విషయాలు నేర్చుకుంటాను. నా స్నేహితులతో కలిసి చదువుతాను. సాయంత్రం ఇంటికి తిరిగి వస్తాను.


In [6]:
#Naive Tokenization
# Telugu paragraph

text = "నేను ప్రతిరోజూ కాలేజీకి వెళ్తాను. అక్కడ నేను కొత్త విషయాలు నేర్చుకుంటాను. నా స్నేహితులతో కలిసి చదువుతాను. సాయంత్రం ఇంటికి తిరిగి వస్తాను."

# Split wherever there is a space
naive_tokens = text.split()

print("Naive tokenization:")
print(naive_tokens)

Naive tokenization:
['నేను', 'ప్రతిరోజూ', 'కాలేజీకి', 'వెళ్తాను.', 'అక్కడ', 'నేను', 'కొత్త', 'విషయాలు', 'నేర్చుకుంటాను.', 'నా', 'స్నేహితులతో', 'కలిసి', 'చదువుతాను.', 'సాయంత్రం', 'ఇంటికి', 'తిరిగి', 'వస్తాను.']


Manually corrected tokens:

["నేను", "ప్రతి", "రోజు", "కాలేజీ", "కి", "వెళ్తాను", ".",
 "అక్కడ", "నేను", "కొత్త", "విషయాలు", "నేర్చుకుంటాను", ".",
 "నా", "స్నేహితులతో", "కలిసి", "చదువుతాను", ".",
 "సాయంత్రం", "ఇంటి", "కి", "తిరిగి", "వస్తాను", "."]


Differences:

ప్రతిరోజూ → ప్రతి + రోజు

కాలేజీకి → కాలేజీ + కి

వెళ్తాను. → వెళ్తాను + .

నేర్చుకుంటాను. → నేర్చుకుంటాను + .

చదువుతాను. → చదువుతాను + .

ఇంటికి → ఇంటి + కి

వస్తాను. → వస్తాను + .

##Compare with an NLP Tool

In [7]:
# Install Indic NLP Library
!pip install indic-nlp-library

In [8]:
from indicnlp.tokenize import indic_tokenize

text = "నేను ప్రతిరోజూ కాలేజీకి వెళ్తాను. అక్కడ నేను కొత్త విషయాలు నేర్చుకుంటాను. నా స్నేహితులతో కలిసి చదువుతాను. సాయంత్రం ఇంటికి తిరిగి వస్తాను."

tool_tokens = indic_tokenize.trivial_tokenize(text)

print("Tool tokenization:")
print(tool_tokens)

Tool tokenization:
['నేను', 'ప్రతిరోజూ', 'కాలేజీకి', 'వెళ్తాను', '.', 'అక్కడ', 'నేను', 'కొత్త', 'విషయాలు', 'నేర్చుకుంటాను', '.', 'నా', 'స్నేహితులతో', 'కలిసి', 'చదువుతాను', '.', 'సాయంత్రం', 'ఇంటికి', 'తిరిగి', 'వస్తాను', '.']


Comparison:

The tool keeps ప్రతిరోజూ as one token, while I split it into ప్రతి + రోజు. The tool also keeps కాలేజీకి and ఇంటికి as one token, while I split their suffixes. Both my manual version and the tool separate punctuation from the words. The manual version is more detailed because it separates some suffixes.

#Q5.3 — Multiword Expressions

Some Telugu MWEs are:

1. ప్రతి రోజు — every day
2. చాలా బాగా — very well
3. ఇంటి దగ్గర — near the house

These can be treated as single units because their words commonly occur together and express one combined meaning.
#Q5.4 — Reflection
The hardest part of Telugu tokenization is separating suffixes and word parts correctly. English is usually easier because spaces often show where words begin and end. Telugu words can contain more information inside a single word, so space-based tokenization is not always enough. Punctuation also needs to be separated from the words. MWEs can make tokenization harder because several words can work together to express one meaning. Overall, Telugu tokenization needs more attention to word structure than simple English space-based tokenization.
